In [1]:
#setup

#conda create --name sds-project python=3.12
#conda install --channel conda-forge pandas numpy matplotlib requests geopandas
#conda install --channel conda-forge ipykernel

import pandas as pd
import requests
import geopandas as gpd
import folium
import numpy as np
from datetime import datetime


In [2]:
MAP_KEY = '5eae605403f5deded880b550afef3667'

def get_transaction_count() :
  count = 0
  try:
    response = requests.get(url = 'https://firms.modaps.eosdis.nasa.gov/mapserver/mapkey_status/?MAP_KEY=' + MAP_KEY)
    data = response.json()
    df = pd.Series(data)
    count = df['current_transactions']
  except:
    print ("Error in our call.")
  return count

In [3]:
#sensors:
da_url = 'https://firms.modaps.eosdis.nasa.gov/api/data_availability/csv/' + MAP_KEY + '/all'
daterange_df = pd.read_csv(da_url)
daterange_df['min_date'] = pd.to_datetime(daterange_df['min_date'], format = '%Y-%m-%d')
daterange_df['max_date'] = pd.to_datetime(daterange_df['max_date'], format = '%Y-%m-%d')

display(daterange_df)

#set one sensor
sensor = daterange_df["data_id"][2] #VIIRS has better resolution than other sensors (375m vs 1000m), NRT means only a few minutes lag
print("Current sensor name: ", sensor)
daterange_df.info()

,data_id,min_date,max_date
0,MODIS_NRT,2026-02-01,2026-05-10
1,MODIS_SP,2000-11-01,2026-01-31
2,VIIRS_NOAA20_NRT,2026-03-01,2026-05-10
3,VIIRS_NOAA20_SP,2018-04-01,2026-02-28
4,VIIRS_NOAA21_NRT,2024-01-17,2026-05-10
5,VIIRS_SNPP_NRT,2026-03-01,2026-05-10
6,VIIRS_SNPP_SP,2012-01-20,2026-02-28
7,LANDSAT_NRT,2022-06-20,2026-05-09
8,GOES_NRT,2022-08-09,2026-05-10
9,BA_MODIS,2000-11-01,2026-02-01


Current sensor name:  VIIRS_NOAA20_NRT
<class 'pandas.DataFrame'>
RangeIndex: 11 entries, 0 to 10
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   data_id   11 non-null     str           
 1   min_date  11 non-null     datetime64[us]
 2   max_date  11 non-null     datetime64[us]
dtypes: datetime64[us](2), str(1)
memory usage: 396.0 bytes


In [4]:
#retrieve data 

#parameters for API call
area = "-20,-37.5,52,37.5" #either "world" or bbox lonmin,latmin,lonmax,latmax, e. g. 0,35,25,70 for europe, "-20,-37.5,52,37.5"  for africa
day_range = 1 #in range(1,5)
enddate = datetime.now() #get todays date
enddate_str = '2026-04-01' #optional: user input for the date
enddate = pd.to_datetime(enddate_str, format = '%Y-%m-%d')


#set sensor parameter based on enddate
#MODIS_NRT where available
#else: MODIS_SP
#error if neither of them is available
mindate_NRT = daterange_df['min_date'][daterange_df['data_id'] == 'VIIRS_NOAA20_NRT'].values[0]
mindate_SP = daterange_df['min_date'][daterange_df['data_id'] == 'VIIRS_NOAA20_SP'].values[0]


request_data = True #flag to prevent data request for invalid enddate

#set sensor based on enddate
if enddate > mindate_NRT:
    sensor = 'VIIRS_NOAA20_NRT'
elif enddate >= mindate_SP:
    sensor = 'VIIRS_NOAA20_SP'
else:
    print(f"Out of date range, please select an end date after {mindate_SP}")
    request_data = False

#data request
if request_data:
    area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + f'/{sensor}/{area}/{day_range}/' + enddate.strftime(format = '%Y-%m-%d')
    start_count = get_transaction_count()
    df_area = pd.read_csv(area_url)
    end_count = get_transaction_count()
    print ('We used %i transactions.' % (end_count-start_count))
    display(df_area.head())
    display(df_area.shape)

We used 8 transactions.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,-31.38113,29.78252,315.22,0.60,0.70,2026-04-01,1,N20,VIIRS,n,2.0NRT,290.85,3.11,N
1,-31.37595,29.78217,312.58,0.60,0.70,2026-04-01,1,N20,VIIRS,n,2.0NRT,290.43,1.86,N
2,-31.37563,29.77569,300.98,0.60,0.70,2026-04-01,1,N20,VIIRS,n,2.0NRT,290.49,1.86,N
3,-29.55105,31.12862,312.82,0.69,0.75,2026-04-01,1,N20,VIIRS,n,2.0NRT,286.55,3.07,N
4,-29.55005,31.13002,318.54,0.70,0.75,2026-04-01,1,N20,VIIRS,n,2.0NRT,286.60,3.77,N


(10250, 14)

In [5]:
#retrieve data from the day before
enddate = enddate - pd.Timedelta(days=1)

#data request
if request_data:
    area_url = 'https://firms.modaps.eosdis.nasa.gov/api/area/csv/' + MAP_KEY + f'/{sensor}/{area}/{day_range}/' + enddate.strftime(format = '%Y-%m-%d')
    start_count = get_transaction_count()
    df_area_t1 = pd.read_csv(area_url)
    end_count = get_transaction_count()
    print ('We used %i transactions.' % (end_count-start_count))
    display(df_area_t1.head())
    display(df_area_t1.shape)

We used 8 transactions.


,latitude,longitude,bright_ti4,scan,track,acq_date,acq_time,satellite,instrument,confidence,version,bright_ti5,frp,daynight
0,33.03976,40.91002,300.45,0.57,0.69,2026-03-31,3,N20,VIIRS,n,2.0NRT,278.20,1.01,N
1,34.72564,33.31570,296.50,0.40,0.44,2026-03-31,3,N20,VIIRS,n,2.0NRT,282.82,0.80,N
2,34.86389,44.01988,332.15,0.75,0.77,2026-03-31,3,N20,VIIRS,n,2.0NRT,279.76,5.51,N
3,34.86433,44.02099,333.83,0.75,0.77,2026-03-31,3,N20,VIIRS,n,2.0NRT,279.90,6.01,N
4,34.89151,43.81254,297.72,0.73,0.76,2026-03-31,3,N20,VIIRS,n,2.0NRT,279.48,24.81,N


(7672, 14)

In [6]:
#clean df_area:

#remove low confidence entries
mask = df_area['confidence'] != "l"
print((df_area['confidence'] == "l").sum(), "Entries removed due to low confidence")
df_area = df_area[mask]

#format datetime
df_area["acq_datetime"] = pd.to_datetime(df_area['acq_date'] + df_area['acq_time'].astype(str).str.zfill(4),  # Zero-pad to 4 digits (e.g., '626' -> '0626')
    format='%Y-%m-%d%H%M', errors='coerce'
)

df_area = df_area.drop(columns = ["acq_date", "acq_time"])

1391 Entries removed due to low confidence


In [7]:
area_gpd = gpd.GeoDataFrame(
    df_area, geometry=gpd.points_from_xy(df_area["longitude"], df_area["latitude"], crs = 4326)
)

vmin = df_area['frp'].quantile(0.02)
vmax = df_area['frp'].quantile(0.98)

#area_gpd.explore(column = "frp", cmap = "YlOrRd", vmin = vmin, vmax = vmax)

area_gpd_sub = area_gpd[['latitude', 'longitude', 'frp', 'acq_datetime', 'geometry']]
area_gpd_sub['acq_datetime'] = area_gpd_sub['acq_datetime'].astype(str)



### fire duration

In [8]:
area_gpd_t1 = gpd.GeoDataFrame(
    df_area_t1, geometry=gpd.points_from_xy(df_area_t1["longitude"], df_area_t1["latitude"], crs = 4326)
)

# Buffer the reference geometry by the distance threshold
distance_threshold = 1000
area_gpd['buffer'] = area_gpd.geometry.to_crs(8857).buffer(distance_threshold).to_crs(4326)
area_gpd = area_gpd.set_geometry('buffer') #make buffer the active geometry columns


# Spatial join to find points within the buffer

area_gpd.head()

# Spatial join with 'cointains' predicate
joined = gpd.sjoin(
    area_gpd,  # Left GeoDataFrame (points/lines/polygons)
    area_gpd_t1,    # Right GeoDataFrame (with 'buffer' column)
    how='left',  # Keep all row from df_area
    predicate='contains',  # Check if area_gpd.geometry is countains area_gpd_t1
    lsuffix='t0',  # Avoid column name conflicts
    rsuffix='t1'
)
joined = joined.drop_duplicates(subset='geometry')  #drop duplicates that were created

# Extract boolean result (True if within buffer)
within_buffer = ~joined['latitude_t1'].isna()  # True if matched (within buffer)

area_gpd = area_gpd.set_geometry('geometry') #make geometry the active geometry column again



area_gpd['fire_t1'] = within_buffer.values
area_gpd.head()


,latitude,longitude,bright_ti4,scan,track,satellite,instrument,confidence,version,bright_ti5,frp,daynight,acq_datetime,geometry,buffer,fire_t1
0,-31.38113,29.78252,315.22,0.60,0.70,N20,VIIRS,n,2.0NRT,290.85,3.11,N,2026-04-01 00:01:00,POINT (29.78252 -31.38113),"POLYGON ((29.79375 -31.38113, 29.79381 -31.381...",True
1,-31.37595,29.78217,312.58,0.60,0.70,N20,VIIRS,n,2.0NRT,290.43,1.86,N,2026-04-01 00:01:00,POINT (29.78217 -31.37595),"POLYGON ((29.7934 -31.37595, 29.79346 -31.3767...",True
2,-31.37563,29.77569,300.98,0.60,0.70,N20,VIIRS,n,2.0NRT,290.49,1.86,N,2026-04-01 00:01:00,POINT (29.77569 -31.37563),"POLYGON ((29.78692 -31.37563, 29.78698 -31.376...",True
3,-29.55105,31.12862,312.82,0.69,0.75,N20,VIIRS,n,2.0NRT,286.55,3.07,N,2026-04-01 00:01:00,POINT (31.12862 -29.55105),"POLYGON ((31.13975 -29.55105, 31.13981 -29.551...",True
4,-29.55005,31.13002,318.54,0.70,0.75,N20,VIIRS,n,2.0NRT,286.60,3.77,N,2026-04-01 00:01:00,POINT (31.13002 -29.55005),"POLYGON ((31.14115 -29.55005, 31.14121 -29.550...",True
